# AE architecture sweep

Stage 1 (capacity, 12 configs) and Stage 2 (optimisation, 9 configs at
the Stage-1 winner) at fixed latent dim 16 on the block-stride 80/10/10
split. Picks the winner by validation MSE-z and persists the trained
winning model to `outputs/checkpoints/ae_sweep_winner.pt`.

Per-stage results are logged to `outputs/sweep_stage1_results.csv` and
`outputs/sweep_stage2_results.csv` incrementally: each row is written as
soon as the config finishes, so an interrupt mid-stage keeps the rows
already completed.

In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Anchor to the repo root so paleoreco imports and relative file paths
# resolve regardless of where Jupyter was launched from.
REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from paleoreco.data import (
    PaleoFieldDataset,
    apply_zscore,
    build_prior_cube,
    compute_zscore_stats,
)
from paleoreco.splits import block_stride_split, summarize_split
from paleoreco.models.autoencoder import ConvAE, count_parameters
from paleoreco.train_ae import set_seed, train
from paleoreco.eval.ae import reconstruct_split
from paleoreco.eval.shared import compute_split_metrics

In [ ]:
SEED = 42
LATENT_DIM = 16
MAX_EPOCHS = 250
PATIENCE = 25
BATCH_SIZE = 32

# Stage 1: capacity grid at the v1-default optimiser settings.
BASE_CHANNELS = (8, 16, 32, 64)
DEPTHS = (2, 3, 4)
LR_STAGE1 = 1e-3
WD_STAGE1 = 1e-4

# Stage 2: optimiser grid at the Stage-1 winning (base, depth).
LRS = (3e-4, 1e-3, 3e-3)
WDS = (1e-5, 1e-4, 1e-3)

OUT_DIR = Path(REPO_ROOT) / "outputs"
CKPT_DIR = OUT_DIR / "checkpoints"
STAGE1_CSV = OUT_DIR / "sweep_stage1_results.csv"
STAGE2_CSV = OUT_DIR / "sweep_stage2_results.csv"
WINNER_PT = CKPT_DIR / "ae_sweep_winner.pt"

OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

## Data and split

In [ ]:
data = build_prior_cube(
    prior_csv=os.path.join(REPO_ROOT, "data/Prior.csv"),
    cache_path=os.path.join(REPO_ROOT, "data/cache/prior_cube.npz"),
)
cube = data["cube"]
ages = data["ages"]
valid = data["valid"]

split = block_stride_split(len(ages))
print(summarize_split(ages, split))

stats = compute_zscore_stats(cube, split["train"], valid)
cube_z = apply_zscore(cube, stats)
mask = stats["safe_valid"]
print(f"\nsafe_valid cells: {int(mask.sum())} / {mask.size}")

In [ ]:
train_dataset = PaleoFieldDataset(cube_z, mask, split["train"])
val_dataset = PaleoFieldDataset(cube_z, mask, split["val"])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"train batches: {len(train_loader)}  val batches: {len(val_loader)}")

## Per-config training helper

In [ ]:
def train_one_config(
    base_channels: int,
    depth: int,
    lr: float,
    weight_decay: float,
) -> dict:
    """Run one training config end-to-end and return its evaluation bundle.

    Sets the seed before model construction so model init is deterministic;
    train() reseeds before its own data-loader iteration so DataLoader
    shuffling is the same across configs.
    """
    set_seed(SEED)
    model = ConvAE(
        latent_dim=LATENT_DIM,
        base_channels=base_channels,
        depth=depth,
    )
    n_params = count_parameters(model)

    t0 = time.perf_counter()
    out = train(
        model,
        train_loader,
        val_loader,
        mask=mask,
        zscore_std=stats["std"],
        lr=lr,
        weight_decay=weight_decay,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        device=device,
        seed=SEED,
        verbose=False,
        progress=True,
    )
    seconds = time.perf_counter() - t0

    # Load best-val weights into the trained model and re-evaluate val to
    # get the full metric bundle (train() history tracks only mse_z / rmse_z).
    model.load_state_dict(out["best_state_dict"])
    truth_z, pred_z = reconstruct_split(
        model, val_dataset, device=device, batch_size=BATCH_SIZE,
    )
    val_metrics = compute_split_metrics(truth_z, pred_z, mask)

    return {
        "state_dict": out["best_state_dict"],
        "val_metrics": val_metrics,
        "n_params": n_params,
        "epochs_trained": out["epochs_trained"],
        "best_epoch": out["best_epoch"],
        "seconds": seconds,
    }

## Stage 1: capacity sweep (12 configs)

Fix `latent_dim=16`, `lr=1e-3`, `weight_decay=1e-4`. Sweep
`base_channels` × `depth` = 4 × 3 grid.

In [ ]:
stage1_rows: list[dict] = []
stage1_configs = [(b, d) for b in BASE_CHANNELS for d in DEPTHS]

for i, (base, depth) in enumerate(stage1_configs, start=1):
    print(f"\n[Stage 1 {i:>2}/{len(stage1_configs)}] base={base}  depth={depth}")
    result = train_one_config(
        base_channels=base, depth=depth,
        lr=LR_STAGE1, weight_decay=WD_STAGE1,
    )
    m = result["val_metrics"]
    row = {
        "base_channels":  base,
        "depth":          depth,
        "val_mse_z":      m["mse_z"],
        "val_rmse_z":     m["rmse_z"],
        "val_rrmse":      m["rrmse"],
        "val_r_squared":  m["r_squared"],
        "val_E_d":        m["E_d"],
        "n_params":       result["n_params"],
        "epochs_trained": result["epochs_trained"],
        "best_epoch":     result["best_epoch"],
    }
    stage1_rows.append(row)
    pd.DataFrame(stage1_rows).to_csv(STAGE1_CSV, index=False)
    print(
        f"  val_mse_z={m['mse_z']:.4f}  R²={m['r_squared']:.3f}  "
        f"E_d={m['E_d']:.3f}  params={result['n_params']:,}  "
        f"epochs={result['epochs_trained']}  ({result['seconds']:.0f}s)"
    )

In [ ]:
stage1_df = pd.DataFrame(stage1_rows).sort_values("val_mse_z").reset_index(drop=True)
print(stage1_df.to_string(index=False))

winner_stage1 = stage1_df.iloc[0]
BASE_STAR = int(winner_stage1["base_channels"])
DEPTH_STAR = int(winner_stage1["depth"])
print(
    f"\nStage-1 winner: base={BASE_STAR}  depth={DEPTH_STAR}  "
    f"val_mse_z={winner_stage1['val_mse_z']:.4f}"
)

## Stage 2: optimisation sweep at the Stage-1 winner (9 configs)

Fix `(base, depth)` at `(BASE_STAR, DEPTH_STAR)`. Sweep `lr` ×
`weight_decay` = 3 × 3 grid. The `(lr=1e-3, weight_decay=1e-4)` cell
duplicates one Stage-1 run; it is re-trained for loop uniformity.

In [ ]:
stage2_rows: list[dict] = []
stage2_states: dict[tuple[float, float], dict] = {}
stage2_configs = [(lr, wd) for lr in LRS for wd in WDS]

for i, (lr, wd) in enumerate(stage2_configs, start=1):
    print(
        f"\n[Stage 2 {i:>2}/{len(stage2_configs)}] "
        f"base={BASE_STAR}  depth={DEPTH_STAR}  "
        f"lr={lr:.0e}  wd={wd:.0e}"
    )
    result = train_one_config(
        base_channels=BASE_STAR, depth=DEPTH_STAR,
        lr=lr, weight_decay=wd,
    )
    m = result["val_metrics"]
    row = {
        "lr":             lr,
        "weight_decay":   wd,
        "val_mse_z":      m["mse_z"],
        "val_rmse_z":     m["rmse_z"],
        "val_rrmse":      m["rrmse"],
        "val_r_squared":  m["r_squared"],
        "val_E_d":        m["E_d"],
        "n_params":       result["n_params"],
        "epochs_trained": result["epochs_trained"],
        "best_epoch":     result["best_epoch"],
    }
    stage2_rows.append(row)
    stage2_states[(lr, wd)] = {
        "state_dict": result["state_dict"],
        "val_mse_z":  m["mse_z"],
        "best_epoch": result["best_epoch"],
    }
    pd.DataFrame(stage2_rows).to_csv(STAGE2_CSV, index=False)
    print(
        f"  val_mse_z={m['mse_z']:.4f}  R²={m['r_squared']:.3f}  "
        f"E_d={m['E_d']:.3f}  epochs={result['epochs_trained']}  "
        f"({result['seconds']:.0f}s)"
    )

In [ ]:
stage2_df = pd.DataFrame(stage2_rows).sort_values("val_mse_z").reset_index(drop=True)
print(stage2_df.to_string(index=False))

winner_stage2 = stage2_df.iloc[0]
LR_STAR = float(winner_stage2["lr"])
WD_STAR = float(winner_stage2["weight_decay"])
WINNER_STATE = stage2_states[(LR_STAR, WD_STAR)]

print(
    f"\nFinal winner: base={BASE_STAR}  depth={DEPTH_STAR}  "
    f"lr={LR_STAR:.0e}  wd={WD_STAR:.0e}\n"
    f"  val_mse_z={WINNER_STATE['val_mse_z']:.4f}  "
    f"best_epoch={WINNER_STATE['best_epoch']}"
)

## Persist winner

Writes `outputs/checkpoints/ae_sweep_winner.pt` with the trained
state\_dict and the config + training metadata. This is the only
checkpoint produced by the sweep.

In [ ]:
checkpoint = {
    "state_dict": WINNER_STATE["state_dict"],
    "config": {
        "latent_dim":    LATENT_DIM,
        "base_channels": BASE_STAR,
        "depth":         DEPTH_STAR,
        "in_channels":   3,
        "out_channels":  2,
        "grid_shape":    (32, 64),
    },
    "training": {
        "lr":           LR_STAR,
        "weight_decay": WD_STAR,
        "max_epochs":   MAX_EPOCHS,
        "patience":     PATIENCE,
        "seed":         SEED,
        "split":        "block_stride_80_10_10",
        "monitor":      "val_mse_z",
    },
    "val_mse_z_at_early_stop": WINNER_STATE["val_mse_z"],
    "best_epoch":              WINNER_STATE["best_epoch"],
}
torch.save(checkpoint, WINNER_PT)
print(f"saved {WINNER_PT}")
print(f"  config:   {checkpoint['config']}")
print(f"  training: {checkpoint['training']}")
print(f"  val_mse_z_at_early_stop: {checkpoint['val_mse_z_at_early_stop']:.4f}")
print(f"  best_epoch: {checkpoint['best_epoch']}")